# Model with NHPP by day of week and response category

In [56]:
from ambdes import SimConfig, ambsys
import pandas as pd
from pathlib import Path

## Suggested data format

Could do wide or long format. For assumption check we want wide. For the use in the model we want long.

Want by category and day of week so can:

* Find proportions C1-C4.
* Check assumption that proportion does not vary notably by day of week.
* Calculate overall arrivals by day.

In [57]:
days = [
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday",
]

arrival_data = {
    "C1": [25, 24, 24, 24, 25, 28, 27],
    "C2": [310, 295, 295, 295, 305, 340, 330],
    "C3": [180, 170, 170, 170, 178, 200, 195],
    "C4": [40, 38, 38, 38, 40, 45, 43],
}

arrivals_per_day = (
    pd.DataFrame(arrival_data, index=days)
    .reset_index(names="day")
    .melt(id_vars="day", var_name="category", value_name="arrivals")
)
arrivals_per_day

,day,category,arrivals
0,monday,C1,25
1,tuesday,C1,24
2,wednesday,C1,24
3,thursday,C1,24
4,friday,C1,25
5,saturday,C1,28
6,sunday,C1,27
7,monday,C2,310
8,tuesday,C2,295
9,wednesday,C2,295


## Assumption: proportion of arrivals by response category don't vary by day of week

In [63]:
arrivals = arrivals_per_day.copy()

# Convert to wide format
arrivals_wide = arrivals.pivot(
    index="day",
    columns="category",
    values="arrivals",
)

# Find proportion of arrivals from each category per day
proportions = arrivals_wide.div(arrivals_wide.sum(axis=1), axis=0)
proportions

category,C1,C2,C3,C4
day,,,,
friday,0.045620,0.556569,0.324818,0.072993
monday,0.045045,0.558559,0.324324,0.072072
saturday,0.045677,0.554649,0.326264,0.073409
sunday,0.045378,0.554622,0.327731,0.072269
thursday,0.045541,0.559772,0.322581,0.072106
tuesday,0.045541,0.559772,0.322581,0.072106
wednesday,0.045541,0.559772,0.322581,0.072106


In [ ]:
# View the variation in proportion across days, by category
summary = pd.DataFrame({
    "mean_prop": proportions.mean(axis=0),
    "min_prop": proportions.min(axis=0),
    "max_prop": proportions.max(axis=0),
    "range": proportions.max(axis=0) - proportions.min(axis=0),
    "sd": proportions.std(axis=0),
})
summary

,mean_prop,min_prop,max_prop,range,sd
category,,,,,
C1,0.045478,0.045045,0.045677,0.000632,0.000212
C2,0.557674,0.554622,0.559772,0.005150,0.002369
C3,0.324411,0.322581,0.327731,0.005150,0.002028
C4,0.072437,0.072072,0.073409,0.001337,0.000539


In [ ]:
# Calculate the maximum absolute deviation from the overall mean proportion
max_abs_dev = proportions.sub(proportions.mean(axis=0), axis=1).abs().max(axis=0)
max_abs_dev

category
C1    0.000433
C2    0.003052
C3    0.003320
C4    0.000972
dtype: float64

In [69]:
# View overall mean proportions
proportions.mean(axis=0)

category
C1    0.045478
C2    0.557674
C3    0.324411
C4    0.072437
dtype: float64

## Class to import arrival dataframe

In [ ]:
class ArrivalConfig:
    def __init__(self, arrivals):
        """Initialise ArrivalConfig.

        Parameters
        ----------
        arrivals : str | Path | pd.DataFrame
            Mean call counts by day, broken down by call category.
        """
        # Import arrivals dataframe
        if isinstance(arrivals, (str, Path)):
            self.arrivals = pd.read_csv(arrivals)
        elif isinstance(arrivals, pd.DataFrame):
            self.arrivals = arrivals
        else:
            raise TypeError(
                f"arrivals must be a file path or DataFrame, "
                f"got {type(arrivals).__name__}."
            )

        # Set the interval duration - 1440 minutes, as daily arrival counts
        self.minutes_per_day = 1440

        # Check the structure of the arrivals dataframe
        self._validate_structure()

        # Find proportions C1-C4 per day

        # Calculate arrival rate per minute
        self.arrivals["rate"] = self.arrivals["arrivals"] / self.minutes_per_day

        # Find maximum arrival rate across all days and categories
        self.lambda_max = self.arrivals["rate"].max()

    def _validate_structure(self):
        """Checks the arrivals dataframe is in the expected structure.

        Check it has all expected columns, entries, etc. TODO: Improve.
        """

        expected_cols = ["day", "category", "arrivals"]
        missing_cols = [c for c in expected_cols if c not in self.arrivals.columns]
        if missing_cols:
            raise ValueError(
                f"arrivals is missing columns: {missing_cols}. "
                f"Expected columns: {expected_cols}"
            )

        expected_days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
        missing_days = [d for d in expected_days if d not in self.arrivals["day"].values]
        if missing_days:
            raise ValueError(
                f"arrivals is missing rows for days: {missing_days}. "
                f"Expected days: {expected_days}."
            )

        expected_cats = ["C1", "C2", "C3", "C4"]
        missing_cats = [c for c in expected_cats if c not in self.arrivals["category"].values]
        if missing_cats:
            raise ValueError(
                f"arrivals is missing rows for categories: {missing_cats}. "
                f"Expected categories: {expected_cats}."
            )

        if (self.arrivals["arrivals"] <= 0).any():
            raise ValueError(
                "All values in the 'arrivals' column must be positive."
            )

        combos = self.arrivals[["day", "category"]]
        if combos.duplicated().any():
            raise ValueError(
                "Each (day, category) pair must appear at most once."
            )

        expected_pairs = {(d,c) for d in expected_days for c in expected_cats}
        actual_pairs = set(map(tuple, combos.to_numpy()))
        missing_pairs = expected_pairs - actual_pairs
        extra_pairs = actual_pairs - expected_pairs

        if missing_pairs:
            raise ValueError(
                "arrivals must contain exactly one row for each "
                "combination of day and category. "
                f"Missing pairs: {sorted(missing_pairs)}."
            )

        if extra_pairs:
            raise ValueError(
                "arrivals contains unexpected (day, category) combinations: "
                f"{sorted(extra_pairs)}."
            )

In [50]:
arrival_config = ArrivalConfig(arrivals=arrivals_per_day)
print(arrival_config.lambda_max)
display(arrival_config.arrivals)

0.2361111111111111


,day,category,arrivals,rate
0,monday,C1,25,0.017361
1,tuesday,C1,24,0.016667
2,wednesday,C1,24,0.016667
3,thursday,C1,24,0.016667
4,friday,C1,25,0.017361
5,saturday,C1,28,0.019444
6,sunday,C1,27,0.018750
7,monday,C2,310,0.215278
8,tuesday,C2,295,0.204861
9,wednesday,C2,295,0.204861


In [22]:
arrival_config["iat"] = arrival_config["arrivals"] / minutes_per_day
arrival_config

,day,category,arrivals,iat
0,monday,C1,25,0.017361
1,tuesday,C1,24,0.016667
2,wednesday,C1,24,0.016667
3,thursday,C1,24,0.016667
4,friday,C1,25,0.017361
5,saturday,C1,28,0.019444
6,sunday,C1,27,0.018750
7,monday,C2,310,0.215278
8,tuesday,C2,295,0.204861
9,wednesday,C2,295,0.204861


In [ ]:
class SimConfig:
    """Configuration for a simulation run.

    Stores input data and run settings used by the model.
    """

    def __init__(
        self,
        ambsys_data,
        arrival_config,
        resource_hours_per_week=52000,
        mean_time_to_scene=10,
        on_scene_time=44,
        mean_time_to_hospital=10,
        wrap_up_time=14,
        warm_up_period=100,
        data_collection_period=100,
        n_reps=5,
    ):
        """Initialise simulation configuration.

        Parameters
        ----------
        ambsys_data : dict
            Input data containing mean and SD of timings for the simulation.
        arrival_config : ArrivalConfig
            TODO.
        resource_hours_per_week : int
            Ambulance resource hours per week.
        mean_time_to_scene : float
            Mean time from ambulance assignment to arrival on scene in minutes.
        on_scene_time : float
            Fixed time in minutes spent on scene before transport.
        mean_time_to_hospital : float
            Mean time from leaving scene to arriving at hospital in minutes.
        wrap_up_time : float
            Fixed time in minutes for post-handover wrap-up before the
            ambulance becomes available again.
        warm_up_period : int
            Duration of the warm-up period in minutes.
        data_collection_period : int
            Duration of the data collection period in minutes.
        n_reps : int
            Number of replications to run.

        """
        # Set up parameters for distributions in required format for
        # sim-tools DistributionsRegistry
        self.dist_config = {
            "call_thinning_exp": {
                "class_name": "Exponential",
                "params": {"mean": self.arrival_config.minutes_per_day / self.arrival_config.lambda_max},
            },
            "call_thinning_uni": {
                "class_name": "Uniform",
                "params": {"low": 0, "high": 1}
            },
            "time_to_scene": {
                "class_name": "Exponential",
                "params": {"mean": mean_time_to_scene},
            },
            "handover_time": {
                "class_name": "Lognormal",
                "params": {
                    "mean": ambsys_data["mean_handover_time_min"],
                    "stdev": ambsys_data["sd_handover_time_min"],
                },
            },
            "time_to_hospital": {
                "class_name": "Exponential",
                "params": {"mean": mean_time_to_hospital},
            },
        }

        # Convert total weekly ambulance-hours into an equivalent constant
        # fleet size, assuming a fixed 24/7 resource pool with no shift
        # pattern. One always-available ambulance provides 168 hours of
        # capacity per week (24 × 7), so we approximate the number of
        # ambulances as resource_hours_per_week / 168.
        self.n_ambulances = round(resource_hours_per_week / 168)

        self.on_scene_time = on_scene_time
        self.wrap_up_time = wrap_up_time
        self.warm_up_period = warm_up_period
        self.data_collection_period = data_collection_period
        self.n_reps = n_reps


In [ ]:
class ConfigValidator:
    """Validates a ``SimConfig`` before a simulation run.

    Checks that all required parameters are present and structurally correct.
    """

    def __init__(self, config):
        """Initialise ConfigValidator.

        Parameters
        ----------
        config : SimConfig
            The configuration object to validate.
        """
        self.config = config

    def validate(self):
        """Run all validation checks.

        Raises
        ------
        ValueError
            If any check fails.
        """
        self._check_arrivals()

    def _check_arrivals(self):
        # TODO
